In [ ]:
import os
import csv
import numpy as np
import nibabel as nib

base_dir = "/Users/avi4336/Desktop/Codex/Python_project_SAFA/dataset/BraiTs2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
out_csv = "/Users/avi4336/Desktop/Codex/Python_project_SAFA/dataset/Data_lodaing/seg_slice_stats.csv"

# make sure output folder exists
os.makedirs(os.path.dirname(out_csv), exist_ok=True)

rows = []
case_dirs = [d for d in os.listdir(base_dir) if d.startswith("BraTS20_")]

for case_id in sorted(case_dirs):
    seg_path = os.path.join(base_dir, case_id, f"{case_id}_seg.nii")
    if not os.path.exists(seg_path):
        continue

    seg = nib.load(seg_path).get_fdata()
    total_slices = seg.shape[2]

    tumor_slice_indices = [z for z in range(total_slices) if np.any(seg[:, :, z] > 0)]
    tumor_slices = len(tumor_slice_indices)

    # store slice list as a string (comma-separated)
    rows.append([
        case_id,
        total_slices,
        tumor_slices,
        ",".join(map(str, tumor_slice_indices))
    ])

# Save CSV
with open(out_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["case_id", "total_slices", "tumor_slices", "tumor_slice_indices"])
    writer.writerows(rows)

print(f"Saved stats for {len(rows)} cases to: {out_csv}")


In [ ]:
import csv

csv_path = "/Users/avi4336/Desktop/Codex/Python_project_SAFA/dataset/Data_lodaing/seg_slice_stats.csv"

total_frames = 0
total_tumor = 0

with open(csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for r in reader:
        total_frames += int(r["total_slices"])
        total_tumor += int(r["tumor_slices"])

total_black = total_frames - total_tumor

print("Total frames:", total_frames)
print("Total tumor slices:", total_tumor)
print("Total black (empty) slices:", total_black)


In [ ]:
import pandas as pd
from sklearn.model_selection import KFold
import os

# Load CSV
df = pd.read_csv("/Users/avi4336/Desktop/Codex/Python_project_SAFA/dataset/Data_lodaing/seg_slice_stats.csv")

# Create output folder
os.makedirs("folds", exist_ok=True)

# Define KFold
k = 4   # ← change here
kf = KFold(n_splits=k, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(df)):

    train_df = df.iloc[train_idx]
    val_df   = df.iloc[val_idx]

    train_df.to_csv(f"folds/fold_{fold+1}_train.csv", index=False)
    val_df.to_csv(f"folds/fold_{fold+1}_val.csv", index=False)

print("4 folds created successfully")